**Yeison Estibel Guerra Soto**

In [0]:

%sql
LIST '/Volumes/workspace/electiva/datos_final/';


In [0]:

# Creación de las tablas Delta del parcial

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.customers
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/customers.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.employee
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/employees.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.orders
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/orders.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.product_categories
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/productcategories.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/productsubcategories.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.products
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/products.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.vendor
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/vendors.csv',
format => 'csv', header => true, inferSchema => true)
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.electiva.vendor_product
USING DELTA AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos_final/vendorproduct.csv',
format => 'csv', header => true, inferSchema => true)
""")

print("Se completó la creación de las ocho tablas Delta.")


In [0]:

%sql
SHOW TABLES IN workspace.electiva;


In [0]:

%sql
DESCRIBE TABLE workspace.electiva.orders;


In [0]:

%sql
SELECT
    CustomerID AS customerid,
    FullName AS fullname,
    CONCAT(
        CustomerID, '-',
        UPPER(LEFT(FirstName, 2)), '-',
        LOWER(LEFT(LastName, 2)), '-AW'
    ) AS user_name
FROM workspace.electiva.customers;



**Resultado:** Se generó el nombre de usuario de cada cliente utilizando su identificación, las dos primeras letras del nombre en mayúsculas, las dos primeras letras del apellido en minúsculas y la terminación AW.


**punto 2**

In [0]:

%sql
CREATE TABLE IF NOT EXISTS workspace.electiva.page_user (
    customerid INT,
    user_name STRING
)
USING DELTA;


In [0]:

%sql
INSERT INTO workspace.electiva.page_user
SELECT
    CustomerID,
    CONCAT(
        CustomerID, '-',
        UPPER(LEFT(FirstName, 2)), '-',
        LOWER(LEFT(LastName, 2)), '-AW'
    )
FROM workspace.electiva.customers;


In [0]:

%sql
SELECT *
FROM workspace.electiva.page_user;



**Resultado:** Se creó la tabla page_user y se almacenaron los nombres de usuario generados para los clientes, permitiendo consultar directamente su identificación y su respectivo usuario.


**punto 3**

In [0]:

%sql
CREATE OR REPLACE VIEW workspace.electiva.v_descuentos_agosto AS
WITH ventas_agosto AS (
    SELECT
        p.ProductName AS productname,
        p.ModelName AS modelname,
        u.user_name,
        o.SalesOrderID AS salesorderid,
        c.CategoryName AS categoryname,
        o.OrderDate AS orderdate,
        o.LineTotal AS linetotal,
        CASE
            WHEN c.CategoryName ILIKE 'Bikes' THEN 0.10
            WHEN c.CategoryName ILIKE 'Clothing' THEN 0.15
            WHEN c.CategoryName ILIKE 'Components' THEN 0.08
            WHEN c.CategoryName ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
    FROM workspace.electiva.orders o
    INNER JOIN workspace.electiva.products p
        ON o.ProductID = p.ProductID
    INNER JOIN workspace.electiva.product_sub_categories s
        ON p.SubCategoryID = s.SubCategoryID
    INNER JOIN workspace.electiva.product_categories c
        ON s.CategoryID = c.CategoryID
    INNER JOIN workspace.electiva.page_user u
        ON o.CustomerID = u.customerid
    WHERE YEAR(TO_DATE(o.OrderDate, 'M/d/yyyy')) = 2011
      AND MONTH(TO_DATE(o.OrderDate, 'M/d/yyyy')) = 8
)
SELECT
    *,
    linetotal - (linetotal * `descuento_mitad_año`)
        AS total_descuentos_aplicados
FROM ventas_agosto;


In [0]:

%sql
CREATE OR REPLACE TABLE workspace.electiva.descuentos_agosto
USING DELTA
AS
SELECT *
FROM workspace.electiva.v_descuentos_agosto;


In [0]:

%sql
SELECT *
FROM workspace.electiva.descuentos_agosto
ORDER BY salesorderid;



**Resultado:** Se identificaron las órdenes realizadas durante agosto de 2011 y se aplicaron los descuentos correspondientes según la categoría de cada producto. También se calculó el valor final de las compras después de aplicar dichos descuentos.


**punto 4**

In [0]:

%sql
CREATE MATERIALIZED VIEW workspace.electiva.customer_class AS
WITH pedidos_clientes AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS cantidad_ordenes
    FROM workspace.electiva.orders
    WHERE YEAR(TO_DATE(OrderDate, 'M/d/yyyy')) = 2011
    GROUP BY CustomerID
)
SELECT
    u.user_name,
    CASE
        WHEN COALESCE(p.cantidad_ordenes, 0) = 3 THEN 'Oro'
        WHEN COALESCE(p.cantidad_ordenes, 0) = 2 THEN 'Plata'
        WHEN COALESCE(p.cantidad_ordenes, 0) < 2 THEN 'Ocasional'
        ELSE 'Sin clasificación'
    END AS user_level,
    c.CustomerID AS customerid,
    c.FullName AS fullname,
    COALESCE(p.cantidad_ordenes, 0) AS cantidad_ordenes
FROM workspace.electiva.customers c
INNER JOIN workspace.electiva.page_user u
    ON c.CustomerID = u.customerid
LEFT JOIN pedidos_clientes p
    ON c.CustomerID = p.CustomerID;


In [0]:

%sql
SELECT *
FROM workspace.electiva.customer_class
ORDER BY cantidad_ordenes DESC;


In [0]:

%sql
SELECT
    user_level,
    COUNT(*) AS total_clientes
FROM workspace.electiva.customer_class
GROUP BY user_level
ORDER BY total_clientes DESC;



**Resultado:** Se clasificaron 635 clientes según sus órdenes realizadas durante 2011. De ellos, 35 pertenecen a la categoría Oro, 131 a Plata y 469 a Ocasional. La clasificación permite identificar a los clientes según su frecuencia de compra.
